# Plate detection — A1 (YOLO26n vs YOLOv8n)
Thin driver (OS-independent). All logic lives in `plate_detect`; this notebook only calls the CLI.

- **Local kernel** with a GPU: repo + prepared `data/` already on disk → the Setup cell just `pip install`s the package.
- **Colab kernel** (free T4): the Setup cell bootstraps everything — clones the private repo (GitHub PAT), installs the CLI, and unzips raw A1 from Google Drive (`MyDrive/UIT_2025/datasets/A1.zip`). All later cells run from the repo root, so the CLI's relative paths resolve.

In [ ]:
# === Setup ===
# LOCAL kernel: repo already on disk + data prepared -> just install the package:
#     %cd /path/to/UIT2026-DoAnCuoiKi
#     !pip install -e src/ml/plate_detection_pipeline
# COLAB kernel: run the bootstrap below (clones private repo BRANCH, installs CLI, pulls raw A1 from Drive).
#   PAT: add a Colab secret named COLAB_PAT (🔑 panel, left sidebar) = your GitHub PAT ('repo' scope).
import os, glob, shutil, getpass

IN_COLAB  = "google.colab" in str(get_ipython())
REPO      = "/content/UIT2026-DoAnCuoiKi"
BRANCH    = "feat/plate-detect-a1"              # package NOT merged to main yet — clone this branch
RAW       = "data/raw/kaggle_vn_plate_segment"  # layout the A1Adapter expects: {images,labels}/{train,val}
DRIVE_ZIP = "/content/drive/MyDrive/UIT_2025/datasets/A1.zip"  # pre-uploaded raw A1 (no re-download)

if IN_COLAB:
    # 1) clone the PRIVATE repo, feature branch. PAT from Colab secret COLAB_PAT (prompt if unset).
    if not os.path.isdir(REPO):
        try:
            from google.colab import userdata
            tok = userdata.get("COLAB_PAT")
        except Exception:
            tok = getpass.getpass("GitHub PAT: ")
        !git clone --branch {BRANCH} --single-branch https://{tok}@github.com/UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi.git {REPO}
        del tok
    %cd {REPO}
    !git rev-parse --abbrev-ref HEAD   # confirm the feature branch is checked out

    # 2) install the package -> puts the `plate_detect` CLI on PATH
    !pip install -q -e src/ml/plate_detection_pipeline

    # 3) raw A1 from the Drive zip (unzip once per runtime, then symlink RAW to it — no 2.6G copy)
    if not os.path.isdir(f"{RAW}/images/train"):
        from google.colab import drive
        drive.mount("/content/drive")
        assert os.path.exists(DRIVE_ZIP), f"{DRIVE_ZIP} not found — upload A1.zip there first"
        !unzip -q -o "{DRIVE_ZIP}" -d /tmp/a1
        hits = glob.glob("/tmp/a1/**/images/train", recursive=True)
        assert hits, "images/train not found after unzip — inspect /tmp/a1 and adjust"
        root = os.path.abspath(hits[0][: -len("/images/train")])
        os.makedirs(os.path.dirname(RAW), exist_ok=True)
        if os.path.islink(RAW) or os.path.exists(RAW):
            (os.unlink if os.path.islink(RAW) else shutil.rmtree)(RAW)
        os.symlink(root, os.path.abspath(RAW))
else:
    # local kernel: assume cwd is the repo root and data/ already present
    !pip install -q -e src/ml/plate_detection_pipeline

# 4) sanity-check the raw layout the adapter reads (train + val, images + labels)
for s in ("train", "val"):
    for k in ("images", "labels"):
        assert os.path.isdir(f"{RAW}/{k}/{s}"), f"missing {RAW}/{k}/{s} — check zip split names (val vs valid)"
print("OK — CLI installed, raw A1 ready at", RAW)

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Prepare (class-map gate → split → dedup train↔test & train↔val → validate)

In [ ]:
!plate_detect prepare

In [ ]:
!plate_detect check

## 2. Train — full matrix @640 (both models × seeds 0,1,2)

In [ ]:
!plate_detect train --imgsz 640 --seeds 0,1,2 --project runs

## 3. imgsz ablation @960 (single seed, both models)

In [ ]:
!plate_detect train --imgsz 960 --seeds 0 --project runs

## 4. Export best → ONNX (per model & imgsz), parity-checked

In [ ]:
# example; repeat per model/imgsz best run:
!plate_detect export --weights runs/yolo26n_s0_640/weights/best.pt --out weights/yolo26n_a1_640.onnx --imgsz 640

## 5. Evaluate on A1 test → comparison table + experiments.csv

In [ ]:
!plate_detect eval --imgszs 640,960 --project runs --weights-dir weights --sample-image data/processed/a1_det/images/test/$(ls data/processed/a1_det/images/test | head -1)